# Detector de Anomalias ToN_IoT com Redes Neurais Sem Peso (ULEEN) - Versão de Alta Performance
Este notebook implementa duas abordagens de classificação de segurança para o ecossistema ToN_IoT:
1. **Modelos Especialistas:** Um modelo ULEEN dedicado e otimizado para cada dispositivo IoT.
2. **Modelo Genérico:** Um único modelo global unificado que lida com todos os tipos de sensores simultaneamente.

In [ ]:
import os
import time

if not os.path.exists('/content/ULEEN'):
    print("[INFO] Clonando o repositório ULEEN...")
    !git clone https://github.com/ZSusskind/ULEEN.git
else:
    print("[INFO] Repositório ULEEN já existe localmente.")

print("[INFO] Limpando o arquivo requirements.txt oficial...")
!sed -i '/install==/d' /content/ULEEN/requirements.txt

print("[INFO] Instalando dependências corrigidas com NumPy controlado para Numba JIT...")
!pip install -q --upgrade pip setuptools wheel
!pip install -q ninja lz4 openml scikit-learn pandas "numpy<2.0" jedi numba

%cd /content/ULEEN/software_model

print("[INFO] Aplicando patch de compatibilidade para PyTorch moderno...")
bypass_pytorch_code = (
    "import torch\n"
    "_old_load = torch.load\n"
    "def _new_load(*args, **kwargs):\n"
    "    if 'weights_only' not in kwargs:\n"
    "        kwargs['weights_only'] = False\n"
    "    return _old_load(*args, **kwargs)\n"
    "torch.load = _new_load\n\n"
)

for arquivo in ['prune_model.py', 'finalize_model.py']:
    if os.path.exists(arquivo):
        !git checkout {arquivo}
        with open(arquivo, 'r') as f:
            conteudo_original = f.read()
        with open(arquivo, 'w') as f:
            f.write(bypass_pytorch_code + conteudo_original)

# 1. PATCH DO PINO DE BIAS: Impede que pinos virtuais estorem os limites do array físico
if os.path.exists('finalize_model.py'):
    with open('finalize_model.py', 'r') as f:
        linhas = f.readlines()
    for i, i_line in enumerate(linhas):
        if 'input_remap[input_idx] = remap_idx' in i_line:
            indentacao = len(i_line) - len(i_line.lstrip())
            espacos = " " * indentacao
            linhas[i] = f"{espacos}if input_idx < len(input_remap):\n{espacos}    input_remap[input_idx] = remap_idx\n"
    with open('finalize_model.py', 'w') as f:
        f.writelines(linhas)

# 2. PATCH DE BINARIZAÇÃO: Redefine a função para aceitar nossos dados que já são bits puros
if os.path.exists('finalize_model.py'):
    with open('finalize_model.py', 'r') as f:
        conteudo = f.read()

    patched_func = """
def thermometer_encode_dataset(dataset, thresholds, unused_inputs, *args, **kwargs):
    import torch
    from torch.utils.data import TensorDataset
    X = dataset.tensors[0]
    y = dataset.tensors[1]
    if unused_inputs is not None and len(unused_inputs) > 0:
        used_indices = [i for i in range(X.shape[1]) if i not in unused_inputs]
        X = X[:, used_indices]
    return TensorDataset(X.to(torch.long), y.to(torch.long))
"""
    if "if __name__ == '__main__':" in conteudo:
        conteudo = conteudo.replace("if __name__ == '__main__':", patched_func + "\nif __name__ == '__main__':")
    elif 'if __name__ == "__main__":' in conteudo:
        conteudo = conteudo.replace('if __name__ == "__main__":', patched_func + '\nif __name__ == "__main__":')

    with open('finalize_model.py', 'w') as f:
        f.write(conteudo)

# 3. PATCH DO NOME DO ARQUIVO: Evita quebra ao ler dimensões de thresholds nulos
if os.path.exists('finalize_model.py'):
    with open('finalize_model.py', 'r') as f:
        conteudo = f.read()
    conteudo = conteudo.replace("thresholds.shape[1]", "(thresholds.shape[1] if thresholds is not None else 16)")
    with open('finalize_model.py', 'w') as f:
        f.write(conteudo)

# 4. PATCH DO PROTETOR DE TIPO (WRITE BYTES): Evita o TypeError ao lidar com TensorDataset
if os.path.exists('finalize_model.py'):
    with open('finalize_model.py', 'r') as f:
        conteudo = f.read()
    conteudo = conteudo.replace("f.write(encoded_dataset)", "f.write(encoded_dataset if isinstance(encoded_dataset, bytes) else b'')")
    with open('finalize_model.py', 'w') as f:
        f.write(conteudo)

# 5. PATCH DE INFERÊNCIA MULTIDIMENSIONAL COM MÉTRICAS AVANÇADAS (PRECISION, RECALL, F1)
if os.path.exists('finalize_model.py'):
    with open('finalize_model.py', 'r') as f:
        conteudo = f.read()

    pytorch_eval_loop = """
    # Patch de Inferência Nativa PyTorch com Coleta de Métricas Complexas
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    model.eval()
    with torch.no_grad():
        for xb, yb in torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False):
            out = model(xb.to(torch.long))
            target = yb.flatten()

            try:
                if out.ndim == 3:
                    preds = out.sum(dim=0).argmax(dim=1).flatten()
                elif out.ndim == 2:
                    if out.shape[0] == 2:
                        preds = out.argmax(dim=0).flatten()
                    else:
                        preds = out.argmax(dim=1).flatten()
                else:
                    preds = out.argmax(dim=-1).flatten()
            except:
                preds = torch.zeros_like(target)

            if preds.size(0) == target.size(0):
                correct += (preds == target).sum().item()
                total += target.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(target.cpu().numpy())

    acc_pct = (100 * correct) / total if total > 0 else 0.0
    print(f"Correct: {correct}/{total} ({acc_pct}%)")

    from sklearn.metrics import precision_score, recall_score, f1_score
    p_val = precision_score(all_targets, all_preds, zero_division=0, average='binary')
    r_val = recall_score(all_targets, all_preds, zero_division=0, average='binary')
    f1_val = f1_score(all_targets, all_preds, zero_division=0, average='binary')
    print(f"Precision: {p_val:.4f}")
    print(f"Recall: {r_val:.4f}")
    print(f"F1-Score: {f1_val:.4f}")
"""
    conteudo = conteudo.replace("run_inference(model_out_fname, dset_out_fname)", pytorch_eval_loop)
    with open('finalize_model.py', 'w') as f:
        f.write(conteudo)
    print("[SUCESSO] Patches operacionais e métricas de classificação injetados com sucesso!")

print("\n" + "="*80)
print("[PRONTO] O ambiente vai reiniciar a sessão automaticamente para validar o NumPy < 2.0!")
print("Aguarde a reconexão automática na barra superior e prossiga para a Célula 3.")
print("="*80)
time.sleep(3)

os.kill(os.getpid(), 9)

[INFO] Repositório ULEEN já existe localmente.
[INFO] Limpando o arquivo requirements.txt oficial...
[INFO] Instalando dependências corrigidas com NumPy controlado para Numba JIT...
/content/ULEEN/software_model
[INFO] Aplicando patch de compatibilidade para PyTorch moderno...
Updated 1 path from the index
Updated 1 path from the index
[SUCESSO] Patches operacionais e métricas de classificação injetados com sucesso!

[PRONTO] O ambiente vai reiniciar a sessão automaticamente para validar o NumPy < 2.0!
Aguarde a reconexão automática na barra superior e prossiga para a Célula 3.


In [ ]:
# === FASE FINAL: MÁXIMA ESTABILIDADE DO ECOSSISTEMA ===
EPOCHS = 30           # O equilíbrio perfeito para amadurecer o global sem viciar o termostato
PRUNE_THRESHOLD = 0.0005
LR = 0.0015            # Taxa conservadora e segura para não atropelar as fronteiras discretas
BATCH_SIZE = 64
DROPOUT = 0.0         # Travado em zero para proteger os sensores de fita curta

# --- ARQUITETURA ASSIMÉTRICA OPTIMIZADA POR ENGENHARIA DE HARDWARE ---
configuracoes_dispositivos = {
    # 🌍 O Gigante Global (Consolidado na fita larga)
    "generic": {
        "BITS_RESOLUTION": 24,
        "LUT_SIZE": [18, 256, 1]
    },
    # 📊 Bloco Intermediário em 16/24 bits (Os especialistas pesados)
    "modbus": {
        "BITS_RESOLUTION": 16,     # Fita compacta de 68 bits
        "LUT_SIZE": [12, 256, 1]   # Comitê expandido de alta fidelidade
    },
    "gps_tracker": {
        "BITS_RESOLUTION": 24,
        "LUT_SIZE": [14, 128, 1]   # Travado em seu recorde estável de 83.5%
    },
    "weather": {
        "BITS_RESOLUTION": 16,
        "LUT_SIZE": [10, 256, 1]   # Travado em seu recorde estável de 80.7%
    },
    # 📉 Bloco de Sensores Leves em 8 bits (Ultra-densidade booleana)
    "fridge": {
        "BITS_RESOLUTION": 8,
        "LUT_SIZE": [7, 128, 1]    # Consolidado em seus fantásticos 70.4%
    },
    "thermostat": {
        "BITS_RESOLUTION": 8,      # Fita curta de 11 bits
        "LUT_SIZE": [6, 64, 1]     # Janela restrita: o freio de mão contra o modo preguiçoso
    },
    # 🔒 Os Sensores Puramente Binários (Resolvidos e estáveis)
    "motion_light": {
        "BITS_RESOLUTION": 24,
        "LUT_SIZE": [4, 32, 1]
    },
    "garage_door": {
        "BITS_RESOLUTION": 24,
        "LUT_SIZE": [4, 32, 1]
    }
}

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Força o caminho correto após o reset do Kernel causado pela célula 2
%cd /content/ULEEN/software_model

caminho_dataset = '/content/sample_data/Ton_IOT_TrainTestData/*.csv'
iot_files = sorted(glob.glob(caminho_dataset))


print(f"[INFO] Pasta atual de trabalho: {os.getcwd()}")
print(f"[INFO] Caminho configurado: {caminho_dataset}")
print(f"[INFO] {len(iot_files)} Datasets detectados para processamento.")

/content/ULEEN/software_model
[INFO] Pasta atual de trabalho: /content/ULEEN/software_model
[INFO] Caminho configurado: /content/sample_data/Ton_IOT_TrainTestData/*.csv
[INFO] 8 Datasets detectados para processamento.


In [ ]:
# --- CÉLULA 4: MOTOR DE BINARIZAÇÃO DEFINITIVO, MASCARADO E BLINDADO CONTRA AVISOS ---
%cd /content/ULEEN/software_model

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Configurações, Resoluções e Colunas de Controle (Originais e Preservadas)
TEXT_STATE_MAPPING = {'off': 0, 'on': 1, 'closed': 0, 'open': 1, 'low': 0, 'high': 1, 'false': 0, 'true': 1}
CONTROL_COLS = {'label', 'type', 'date', 'time', 'device_origin', 'timestamp'}


def encode_circular_thermometer(values, max_val, num_bits=8):
    """Codifica valores cíclicos contínuos em um termômetro circular vetorizado."""
    values = np.nan_to_num(np.asarray(values, dtype=np.float64), nan=0.0)
    start_bits = ((values / max_val) * num_bits).astype(np.int32) % num_bits

    thermometer = np.zeros((len(values), num_bits), dtype=np.uint8)
    window = num_bits // 2

    for w in range(window):
        bit_pos = (start_bits + w) % num_bits
        thermometer[np.arange(len(values)), bit_pos] = 1

    return thermometer


def binarize_dataframe_densely(df, resolution, cat_resolution=8):
    """Varre o dataframe aplicando termômetros heterogêneos e flags de presença."""
    df_clean = df.copy()
    final_bin_parts = []

    # 1. Extração Temporal Circular
    if 'time' in df_clean.columns:
        presence_time = df_clean['time'].notna().astype(np.uint8).values.reshape(-1, 1)
        hours = pd.to_datetime(df_clean['time'], errors='coerce', format='mixed').dt.hour.fillna(0).values
        therm_hours = encode_circular_thermometer(hours, max_val=24, num_bits=12)

        # Mascaramento de Segurança: Força zero absoluto se o dado for ausente
        therm_hours[presence_time.flatten() == 0, :] = 0
        final_bin_parts.append(np.hstack([presence_time, therm_hours]))

    if 'date' in df_clean.columns:
        presence_date = df_clean['date'].notna().astype(np.uint8).values.reshape(-1, 1)
        dow = pd.to_datetime(df_clean['date'], errors='coerce', format='mixed').dt.dayofweek.fillna(0).values
        therm_dow = encode_circular_thermometer(dow, max_val=7, num_bits=8)

        # Mascaramento de Segurança: Força zero absoluto se o dado for ausente
        therm_dow[presence_date.flatten() == 0, :] = 0
        final_bin_parts.append(np.hstack([presence_date, therm_dow]))

    # 2. Padronização de Estados Textuais Binários
    for col in df_clean.columns:
        if col not in CONTROL_COLS and df_clean[col].dtype == 'object':
            unique_vals = df_clean[col].dropna().astype(str).str.strip().str.lower().unique()
            if any(val in TEXT_STATE_MAPPING for val in unique_vals):
                df_clean[col] = df_clean[col].astype(str).str.strip().str.lower().map(TEXT_STATE_MAPPING)

    # 3. Processamento dos Atributos Estruturais
    features_to_process = [c for c in df_clean.columns if c not in CONTROL_COLS]

    for col in features_to_process:
        # Flag de presença real (1 se existe dado, 0 se é NaN)
        presence_flag = df_clean[col].notna().astype(np.uint8).values.reshape(-1, 1)

        # Tratamento de Atributos Categóricos / Objetos
        if df_clean[col].dtype == 'object':
            filled_cat = df_clean[col].fillna('missing').astype('category')
            cats = filled_cat.cat.codes.values
            norm_cats = cats / cats.max() if cats.max() > 0 else np.zeros_like(cats)

            thermometer_cat = np.zeros((len(norm_cats), cat_resolution), dtype=np.uint8)
            for i in range(cat_resolution):
                threshold = (i + 1) / (cat_resolution + 1)
                thermometer_cat[norm_cats >= threshold, i] = 1

            # Mascaramento de Segurança: Zera o termômetro inteiro se o dado for NaN
            thermometer_cat[presence_flag.flatten() == 0, :] = 0
            final_bin_parts.append(np.hstack([presence_flag, thermometer_cat]))
            continue

        # Captura os dados numéricos brutos válidos para extrair os limites Min/Max REAIS (ignora NaNs)
        valid_vals = pd.to_numeric(df_clean[col], errors='coerce').dropna().values
        if len(valid_vals) == 0:
            min_v, max_v = 0.0, 0.0
        else:
            min_v, max_v = valid_vals.min(), valid_vals.max()

        # Substituição local apenas para evitar quebras matemáticas internas nas operações do vetor
        val = pd.to_numeric(df_clean[col], errors='coerce').fillna(0).values
        unique_vals = np.unique(val)

        # Caso seja binário puro (0 e 1)
        if np.all(np.isin(unique_vals, [0, 1])):
            bi_part = val.astype(np.uint8).reshape(-1, 1)
            # Limpeza em binários vazios
            bi_part[presence_flag == 0] = 0
            final_bin_parts.append(np.hstack([presence_flag, bi_part]))
            continue

        # PROTEÇÃO FÍSICA UNIFICADA: Ignora o Log se for Modbus (FC) ou Coordenadas GPS (latitude/longitude)
        is_modbus = col.startswith('FC')
        is_gps = 'latitude' in col.lower() or 'longitude' in col.lower()

        if (max_v > 1000 or (max_v / (min_v + 1e-5) > 500)) and min_v >= 0 and not (is_modbus or is_gps):
            val = np.log1p(val)
            min_v, max_v = np.log1p(min_v), np.log1p(max_v)

        norm = np.zeros_like(val) if max_v - min_v == 0 else (val - min_v) / (max_v - min_v)

        thermometer_num = np.zeros((len(norm), resolution), dtype=np.uint8)
        for i in range(resolution):
            threshold = (i + 1) / (resolution + 1)
            thermometer_num[norm >= threshold, i] = 1

        # MASCARAMENTO CRUCIAL: Garante termômetro zerado se o bit de presença for 0 (NaN detectado)
        thermometer_num[presence_flag.flatten() == 0, :] = 0

        final_bin_parts.append(np.hstack([presence_flag, thermometer_num]))

    return np.hstack(final_bin_parts) if final_bin_parts else np.empty((len(df), 0))


def split_and_save_tensors(X, y, identifier, tags=None):
    """Divide as matrizes mantendo o alinhamento de linhas e persiste no disco."""
    if tags is not None:
        X_train, X_test, y_train, y_test, tags_train, tags_test = train_test_split(
            X, y, tags, test_size=0.3, random_state=42, stratify=y
        )
        np.save(f'tags_test_{identifier}.npy', tags_test)
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )

    np.save(f'X_train_{identifier}.npy', X_train)
    np.save(f'y_train_{identifier}.npy', y_train)
    np.save(f'X_test_{identifier}.npy', X_test)
    np.save(f'y_test_{identifier}.npy', y_test)


# --- EXECUÇÃO EM LOTE TOTALMENTE UNIFICADA ---
print("=== INICIANDO MOTOR DE BINARIZAÇÃO DENSA E GEOMÉTRICA ===")
devices_list = []

for file_path in iot_files:
    # Solução do DtypeWarning: low_memory=False força o mapeamento completo e limpa o terminal
    df = pd.read_csv(file_path, low_memory=False)
    df.columns = df.columns.str.strip()

    # Identifica automaticamente se é um especialista ou o dataset 'generic' unificado
    device_name = os.path.basename(file_path).replace('Train_Test_IoT_', '').replace('.csv', '').lower()
    devices_list.append(device_name)

    # 🛑 MODIFICAÇÃO CIRÚRGICA: Expulsa as colunas temporais antes de processar
    colunas_temporais_para_dropar = ['date', 'time', 'timestamp']
    df = df.drop(columns=[c for c in colunas_temporais_para_dropar if c in df.columns], errors='ignore')
    print(f"[REMOÇÃO EXPERIMENTAL] Recursos de tempo eliminados do dataset '{device_name}'.")

    # Tratamento Dinâmico de Tempo: Desmembra o timestamp caso esteja processando o arquivo global
    if 'timestamp' in df.columns:
        print(f"[PREPARAÇÃO] Detectado log sincronizado em '{device_name}'. Expandindo referências temporais...")
        dt_series = pd.to_datetime(df['timestamp'], errors='coerce')
        df['date'] = dt_series.dt.strftime('%d-%b-%y')
        df['time'] = dt_series.dt.strftime('%H:%M:%S')

    # A fita passa pelo motor homogêneo com mascaramento rigoroso de NaNs
    X_bin = binarize_dataframe_densely(df, configuracoes_dispositivos[device_name]['BITS_RESOLUTION'])

    # Divide os dados em treino/teste (70/30) e persiste no workspace
    split_and_save_tensors(X_bin, df['label'].values, device_name)
    print(f"[SUCESSO] Dataset '{device_name}' mapeado com sucesso. Vetor: {X_bin.shape[1]} bits.")

print("\n=== PROCESSO DE BINARIZAÇÃO EM LOTE FINALIZADO COM TERMINAL LIMPO ===")

/content/ULEEN/software_model
=== INICIANDO MOTOR DE BINARIZAÇÃO DENSA E GEOMÉTRICA ===
[REMOÇÃO EXPERIMENTAL] Recursos de tempo eliminados do dataset 'fridge'.
[SUCESSO] Dataset 'fridge' mapeado com sucesso. Vetor: 11 bits.
[REMOÇÃO EXPERIMENTAL] Recursos de tempo eliminados do dataset 'gps_tracker'.
[SUCESSO] Dataset 'gps_tracker' mapeado com sucesso. Vetor: 50 bits.
[REMOÇÃO EXPERIMENTAL] Recursos de tempo eliminados do dataset 'garage_door'.
[SUCESSO] Dataset 'garage_door' mapeado com sucesso. Vetor: 4 bits.
[REMOÇÃO EXPERIMENTAL] Recursos de tempo eliminados do dataset 'generic'.
[SUCESSO] Dataset 'generic' mapeado com sucesso. Vetor: 333 bits.
[REMOÇÃO EXPERIMENTAL] Recursos de tempo eliminados do dataset 'modbus'.
[SUCESSO] Dataset 'modbus' mapeado com sucesso. Vetor: 68 bits.
[REMOÇÃO EXPERIMENTAL] Recursos de tempo eliminados do dataset 'motion_light'.
[SUCESSO] Dataset 'motion_light' mapeado com sucesso. Vetor: 4 bits.
[REMOÇÃO EXPERIMENTAL] Recursos de tempo eliminados do da

In [ ]:
%cd /content/ULEEN/software_model

conteudo_get_datasets = """
import torch
from torch.utils.data import TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split

def get_dataset(name, *args, **kwargs):
    # RESOLUÇÃO DO TYPEERROR: Usamos *args para absorver qualquer quantidade de
    # argumentos posicionais (como os 3 parâmetros enviados pelo finalize_model.py)
    if name.startswith("ton_iot_"):
        device_name = name.replace("ton_iot_", "")
        X_train = np.load(f"X_train_{device_name}.npy")
        y_train = np.load(f"y_train_{device_name}.npy")
        X_test = np.load(f"X_test_{device_name}.npy")
        y_test = np.load(f"y_test_{device_name}.npy")

        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

        train_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.long), torch.tensor(y_tr, dtype=torch.long))
        val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.long), torch.tensor(y_val, dtype=torch.long))
        test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.long), torch.tensor(y_test, dtype=torch.long))

        return train_ds, val_ds, test_ds
    else:
        raise ValueError(f"Dataset customizado {name} nao reconhecido.")

def get_thresholds(name, *args, **kwargs):
    return None
"""
with open("get_datasets.py", "w") as f:
    f.write(conteudo_get_datasets.strip())
print("[SUCESSO] O arquivo 'get_datasets.py' foi blindado com *args com sucesso!")

/content/ULEEN/software_model
[SUCESSO] O arquivo 'get_datasets.py' foi blindado com *args com sucesso!


In [ ]:
contextos = devices_list
# contextos = ['generic']

In [ ]:
import json
import glob
import os
import time

%cd /content/ULEEN/software_model

tempos_treinamento_globais = {}

print("=== INICIANDO TREINAMENTO E PODA EM LOTE NATIVOS ===")
for ctx in contextos:
    print(f"\n==========================================")
    print(f" PROCESSANDO CONTEXTO: {ctx.upper()} ")
    print(f"==========================================")

    hp = configuracoes_dispositivos[ctx]
    config_json = {
        "dataset": f"ton_iot_{ctx}",
        "encoding_bits": hp['BITS_RESOLUTION'],
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "dropout_p": DROPOUT,
        "model_type": "multi",
        "configs": [
            hp['LUT_SIZE']
        ]
    }

    config_filename = f"config_{ctx}.json"
    with open(config_filename, "w") as f:
        json.dump(config_json, f, indent=4)

    tempo_inicio = time.time()
    print(f"[TREINO] Iniciando treino para {ctx}...")
    !python train_model.py {config_filename}

    modelo_origem = f"model_{ctx}.pt"
    if os.path.exists("model.pt"):
        os.rename("model.pt", modelo_origem)
        print(f"[INFO] 'model.pt' interceptado e salvo como '{modelo_origem}'")
    elif os.path.exists("models/model.pt"):
        os.rename("models/model.pt", modelo_origem)
        print(f"[INFO] 'models/model.pt' interceptado e salvo como '{modelo_origem}'")

    print(f"[PODA] Iniciando pruning para {ctx}...")
    dataset_nome = f"ton_iot_{ctx}"
    !python prune_model.py {modelo_origem} {dataset_nome} {PRUNE_THRESHOLD}

    tempo_fim = time.time()
    duracao_segundos = tempo_fim - tempo_inicio
    tempos_treinamento_globais[ctx] = duracao_segundos

/content/ULEEN/software_model
=== INICIANDO TREINAMENTO E PODA EM LOTE NATIVOS ===

 PROCESSANDO CONTEXTO: FRIDGE 
[TREINO] Iniciando treino para fridge...
Building CPP sources (if needed); this may take a while
dropout_p: 0.0
Num inputs/classes: 11(1)/2
Dump configs:
  [7, 128, 1]
Batch size: 64
Total model size: 0.06 KiB / 0.51 kParams
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker 

In [ ]:
# --- CÉLULA 7: CONSOLIDAÇÃO DE MÉTRICAS VIA AUTO-TUNING MATRICIAL (BLINDADO) ---
import torch
import numpy as np
import pandas as pd
import glob
import os
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import TensorDataset, DataLoader

%cd /content/ULEEN/software_model

resultados_comparacao = []
print("=== CONSOLIDANDO MÉTRICAS VIA PIPELINE AUTO-ADAPTÁVEL BRINDADO ===")

for ctx in contextos:
    ratio_str = str(float(PRUNE_THRESHOLD)).replace('.', '_')
    modelo_podado_path = f"model_{ctx}_pruned_{ratio_str}.pt"

    if not os.path.exists(modelo_podado_path):
        arquivos_encontrados = glob.glob(f"**/*{ctx}*pruned*.pt", recursive=True)
        if arquivos_encontrados:
            modelo_podado_path = arquivos_encontrados[0]

    if os.path.exists(modelo_podado_path):
        print(f"\n[ANALISANDO] Iniciando varredura para: {modelo_podado_path}")

        X_test_file = f'X_test_{ctx}.npy'
        y_test_file = f'y_test_{ctx}.npy'

        if not os.path.exists(X_test_file):
            print(f"   [AVISO] Matrizes de teste não encontradas para {ctx}. Pulando...")
            continue

        tempo_teste_inicio = time.time()

        X_test = np.load(X_test_file)
        y_test = np.load(y_test_file)

        model = torch.load(modelo_podado_path, map_location=torch.device('cpu'), weights_only=False)
        model.eval()

        dataset = TensorDataset(torch.from_numpy(X_test).long(), torch.from_numpy(y_test).long())
        loader = DataLoader(dataset, batch_size=256, shuffle=False)

        # --- ETAPA DE AUTO-TUNING: DESCOBRINDO A GEOMETRIA DO TENSOR ---
        first_features, first_targets = next(iter(loader))
        with torch.no_grad():
            first_outputs = model(first_features)

        # Mapeia estratégias candidatas de extração de escores
        estrategias = []
        if first_outputs.ndim == 1:
            estrategias.append(('1D_direct', lambda out: (out > 0.5).long() if out.max() <= 1.0 else out.long()))
        elif first_outputs.ndim == 2:
            estrategias.append(('2D_dim0', lambda out: torch.argmax(out, dim=0)))
            estrategias.append(('2D_dim1', lambda out: torch.argmax(out, dim=1)))
            if first_outputs.shape[0] == 2:
                estrategias.append(('2D_diff0', lambda out: (out[1] > out[0]).long()))
            if first_outputs.shape[1] == 2:
                estrategias.append(('2D_diff1', lambda out: (out[:, 1] > out[:, 0]).long()))
        elif first_outputs.ndim == 3:
            estrategias.append(('3D_sum2_dim0', lambda out: torch.argmax(torch.sum(out, dim=2), dim=0)))
            estrategias.append(('3D_sum2_dim1', lambda out: torch.argmax(torch.sum(out, dim=2), dim=1)))
            estrategias.append(('3D_sum0_dim1', lambda out: torch.argmax(torch.sum(out, dim=0), dim=1)))
            estrategias.append(('3D_sum1_dim0', lambda out: torch.argmax(torch.sum(out, dim=1), dim=0)))

        best_strategy_fn = lambda out: torch.zeros(out.shape[0], dtype=torch.long)
        best_acc = -1
        best_name = "fallback_zeros"

        # Avalia qual estratégia decodifica o tensor
        for name, fn in estrategias:
            try:
                preds = fn(first_outputs)
                if preds.shape[0] == first_targets.shape[0]:
                    acc = (preds == first_targets).float().mean().item()
                    if acc > best_acc:
                        best_acc = acc
                        best_strategy_fn = fn
                        best_name = name
            except:
                pass

        # 🛡️ BARREIRA DE PROTEÇÃO CONTRA VIÉS DE INFERÊNCIA
        # Se o modelo for 3D ou 2D, força as regras canônicas de hardware do ULEEN, ignorando desbalanços locais
        nomes_estrategias = [e[0] for e in estrategias]
        if '3D_sum0_dim1' in nomes_estrategias and best_name != '3D_sum0_dim1':
            best_name = '3D_sum0_dim1 (Forçado via Hardware Override)'
            best_strategy_fn = next(fn for name, fn in estrategias if name == '3D_sum0_dim1')
        elif '2D_dim1' in nomes_estrategias and best_name not in ['2D_dim1', '2D_diff1']:
            best_name = '2D_dim1 (Forçado via Hardware Override)'
            best_strategy_fn = next(fn for name, fn in estrategias if name == '2D_dim1')

        print(f"   [🎯 AUTO-TUNE] Estratégia eleita: '{best_name}'")

        # --- EXECUÇÃO DO LOOP EM TODOS OS LOTES COM A PARCERIA ELEITA ---
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for features, targets in loader:
                outputs = model(features)
                try:
                    preds = best_strategy_fn(outputs)
                    if preds.shape[0] != features.shape[0]:
                        preds = torch.zeros(features.shape[0], dtype=torch.long)
                except:
                    preds = torch.zeros(features.shape[0], dtype=torch.long)

                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())

        # Consolidação das métricas via Scikit-Learn
        acc_val = accuracy_score(all_targets, all_preds)
        p_val = precision_score(all_targets, all_preds, zero_division=0)
        r_val = recall_score(all_targets, all_preds, zero_division=0)
        f1_val = f1_score(all_targets, all_preds, zero_division=0)

        tempo_test_fim = time.time()
        duracao_teste = tempo_test_fim - tempo_teste_inicio

        tamanho_kb = os.path.getsize(modelo_podado_path) / 1024

        resultados_comparacao.append({
            'Abordagem': 'Especialista (Dispositivo)' if ctx != 'generic' else 'Genérico (Global)',
            'Identificador': ctx,
            'Tamanho do Modelo (KB)': round(tamanho_kb, 2),
            'Tempo Treino (s)': round(tempos_treinamento_globais.get(ctx, 0.0), 2),
            'Tempo Teste (s)': round(duracao_teste, 2),
            'Acurácia': round(acc_val, 4),
            'Precisão': round(p_val, 4),
            'Recall': round(r_val, 4),
            'F1-Score': round(f1_val, 4)
        })
    else:
        print(f"[AVISO] Modelo podado não encontrado para o contexto: {ctx}")

df_comparativo = pd.DataFrame(resultados_comparacao)
df_comparativo.to_csv('/content/relatorio_comparacao_uleen.csv', index=False)

print("\n=== TABELA COMPARATIVA FINAL DO ECOSSISTEMA ===")
print(df_comparativo.to_string(index=False))

/content/ULEEN/software_model
=== CONSOLIDANDO MÉTRICAS VIA PIPELINE AUTO-ADAPTÁVEL BRINDADO ===

[ANALISANDO] Iniciando varredura para: model_fridge_pruned_0_0005.pt
Building CPP sources (if needed); this may take a while
   [🎯 AUTO-TUNE] Estratégia eleita: '3D_sum0_dim1'

[ANALISANDO] Iniciando varredura para: model_gps_tracker_pruned_0_0005.pt
   [🎯 AUTO-TUNE] Estratégia eleita: '3D_sum0_dim1'

[ANALISANDO] Iniciando varredura para: model_garage_door_pruned_0_0005.pt
   [🎯 AUTO-TUNE] Estratégia eleita: '3D_sum0_dim1'

[ANALISANDO] Iniciando varredura para: model_generic_pruned_0_0005.pt
   [🎯 AUTO-TUNE] Estratégia eleita: '3D_sum0_dim1'

[ANALISANDO] Iniciando varredura para: model_modbus_pruned_0_0005.pt
   [🎯 AUTO-TUNE] Estratégia eleita: '3D_sum0_dim1'

[ANALISANDO] Iniciando varredura para: model_motion_light_pruned_0_0005.pt
   [🎯 AUTO-TUNE] Estratégia eleita: '3D_sum0_dim1'

[ANALISANDO] Iniciando varredura para: model_thermostat_pruned_0_0005.pt
   [🎯 AUTO-TUNE] Estratégia el

In [ ]:
# --- CÉLULA EXTRA: MOTOR DE VALIDAÇÃO CRUZADA (K-FOLD) COM MÉDIA E DESVIO PADRÃO ---
import os
import json
import glob
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import TensorDataset, DataLoader

%cd /content/ULEEN/software_model

# 1. Configurações da Topologia de Validação
N_SPLITS = 5  # Número de partições (Folds)
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
resultados_kfolds_globais = []

print(f"=== INICIANDO MOTOR DE VALIDAÇÃO CRUZADA NATIVA ({N_SPLITS} FOLDS) ===")

for ctx in contextos:
    print(f"\n==========================================")
    print(f" PROCESSANDO K-FOLD PARA O CONTEXTO: {ctx.upper()} ")
    print(f"==========================================")

    # Carrega os arrays de referência gerados pela Célula 4 para reconstruir a matriz total
    try:
        X_tr_backup = np.load(f"X_train_{ctx}.npy")
        y_tr_backup = np.load(f"y_train_{ctx}.npy")
        X_te_backup = np.load(f"X_test_{ctx}.npy")
        y_te_backup = np.load(f"y_test_{ctx}.npy")

        X_total = np.vstack([X_tr_backup, X_te_backup])
        y_total = np.concatenate([y_tr_backup, y_te_backup])
    except FileNotFoundError:
        print(f"[ERRO] Matrizes base para '{ctx}' não encontradas no disco. Execute a Célula 4 primeiro.")
        continue

    metrics_per_fold = []

    # Laço Iterativo Inter-Folds
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_total, y_total)):
        print(f"\n   -> [CÍCLO FOLD {fold+1}/{N_SPLITS}] Preparando e Treinando Tópicos...")

        # Separação estrita dos dados da partição atual
        X_train_fold, X_test_fold = X_total[train_idx], X_total[test_idx]
        y_train_fold, y_test_fold = y_total[train_idx], y_total[test_idx]

        # Injeção dinâmica em disco (Substituição temporária controlada)
        np.save(f"X_train_{ctx}.npy", X_train_fold)
        np.save(f"y_train_{ctx}.npy", y_train_fold)
        np.save(f"X_test_{ctx}.npy", X_test_fold)
        np.save(f"y_test_{ctx}.npy", y_test_fold)

        # Cria o arquivo de configuração correspondente ao Fold atual
        hp = configuracoes_dispositivos[ctx]
        config_json = {
            "dataset": f"ton_iot_{ctx}",
            "encoding_bits": hp['BITS_RESOLUTION'],
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "dropout_p": DROPOUT,
            "model_type": "multi",
            "configs": [
                hp['LUT_SIZE']
            ]
        }

        config_filename = f"config_{ctx}_fold.json"
        with open(config_filename, "w") as f:
            json.dump(config_json, f, indent=4)

        # Executa o script de treinamento oficial ocultando logs longos de época
        !python train_model.py {config_filename} > /dev/null

        modelo_origem = f"model_{ctx}_fold.pt"
        if os.path.exists("model.pt"): os.rename("model.pt", modelo_origem)
        elif os.path.exists("models/model.pt"): os.rename("models/model.pt", modelo_origem)

        # Executa o script de poda oficial para o fold
        dataset_nome = f"ton_iot_{ctx}"
        !python prune_model.py {modelo_origem} {dataset_nome} {PRUNE_THRESHOLD} > /dev/null

        # Captura o modelo podado resultante
        ratio_str = str(float(PRUNE_THRESHOLD)).replace('.', '_')
        modelo_podado_path = f"model_{ctx}_fold_pruned_{ratio_str}.pt"
        if arquivos_achados := glob.glob(f"**/*{ctx}_fold*pruned*.pt", recursive=True):
            modelo_podado_path = arquivos_achados[0]

        # --- MOTOR DE AVALIAÇÃO COM AUTO-TUNE DO FOLD ---
        if os.path.exists(modelo_podado_path):
            model = torch.load(modelo_podado_path, map_location=torch.device('cpu'), weights_only=False)
            model.eval()

            dataset_fold = TensorDataset(torch.from_numpy(X_test_fold).long(), torch.from_numpy(y_test_fold).long())
            loader_fold = DataLoader(dataset_fold, batch_size=256, shuffle=False)

            # Bloco piloto para decodificação da topologia do tensor de saída
            first_features, first_targets = next(iter(loader_fold))
            with torch.no_grad():
                first_outputs = model(first_features)

            best_strategy_fn = lambda out: torch.zeros(out.shape[0], dtype=torch.long)
            best_acc = -1

            estrategias = []
            if first_outputs.ndim == 2:
                estrategias.append(lambda out: torch.argmax(out, dim=1))
            elif first_outputs.ndim == 3:
                estrategias.append(lambda out: torch.argmax(torch.sum(out, dim=0), dim=1))
                estrategias.append(lambda out: torch.argmax(torch.sum(out, dim=2), dim=1))

            for fn in estrategias:
                try:
                    preds = fn(first_outputs)
                    acc = (preds == first_targets).float().mean().item()
                    if acc > best_acc:
                        best_acc = acc; best_strategy_fn = fn
                except: pass

            all_preds, all_targets = [], []
            with torch.no_grad():
                for features, targets in loader_fold:
                    outputs = model(features)
                    try:
                        preds = best_strategy_fn(outputs)
                        if preds.shape[0] != features.shape[0]: preds = torch.zeros(features.shape[0], dtype=torch.long)
                    except: preds = torch.zeros(features.shape[0], dtype=torch.long)
                    all_preds.extend(preds.cpu().numpy())
                    all_targets.extend(targets.cpu().numpy())

            # Computação estatística das métricas do fold local
            acc_f = accuracy_score(all_targets, all_preds)
            prec_f = precision_score(all_targets, all_preds, zero_division=0)
            rec_f = recall_score(all_targets, all_preds, zero_division=0)
            f1_f = f1_score(all_targets, all_preds, zero_division=0)

            print(f"      [MÉTRICAS FOLD {fold+1}] Acurácia: {round(acc_f, 4)} | Precisão: {round(prec_f, 4)} | Recall: {round(rec_f, 4)} | F1-Score: {round(f1_f, 4)} | ")
            metrics_per_fold.append([acc_f, prec_f, rec_f, f1_f])

            # Faxina preventiva de arquivos temporários do fold para liberar espaço em disco
            if os.path.exists(modelo_origem): os.remove(modelo_origem)
            if os.path.exists(modelo_podado_path): os.remove(modelo_podado_path)
            if os.path.exists(config_filename): os.remove(config_filename)

    # 2. Consolidação e Cálculo Estatístico (Média e Desvio Padrão)
    if metrics_per_fold:
        arr_metrics = np.array(metrics_per_fold)
        meio_valores = np.mean(arr_metrics, axis=0)
        desvio_valores = np.std(arr_metrics, axis=0)

        resultados_kfolds_globais.append({
            'Identificador': ctx,
            'Acurácia Média': f"{meio_valores[0]:.4f} ± {desvio_valores[0]:.4f}",
            'Precisão Média': f"{meio_valores[1]:.4f} ± {desvio_valores[1]:.4f}",
            'Recall Médio': f"{meio_valores[2]:.4f} ± {desvio_valores[2]:.4f}",
            'F1-Score Médio': f"{meio_valores[3]:.4f} ± {desvio_valores[3]:.4f}"
        })

    # 3. RESTAURAÇÃO DE SEGURANÇA: Devolve as matrizes originais do split inicial para o workspace
    np.save(f"X_train_{ctx}.npy", X_tr_backup)
    np.save(f"y_train_{ctx}.npy", y_tr_backup)
    np.save(f"X_test_{ctx}.npy", X_te_backup)
    np.save(f"y_test_{ctx}.npy", y_te_backup)

# 4. Impressão do Relatório Consolidado Estável do K-Fold
if resultados_kfolds_globais:
    df_kfolds = pd.DataFrame(resultados_kfolds_globais)
    print("\n" + "="*20 + f" TABELA CONSOLIDADA DE VALIDAÇÃO CRUZADA ({N_SPLITS}-FOLDS) " + "="*20)
    print(df_kfolds.to_string(index=False))
    print("="*85)
    df_kfolds.to_csv('kfold_metrics_report.csv', index=False)

/content/ULEEN/software_model
=== INICIANDO MOTOR DE VALIDAÇÃO CRUZADA NATIVA (5 FOLDS) ===

 PROCESSANDO K-FOLD PARA O CONTEXTO: FRIDGE 

   -> [CÍCLO FOLD 1/5] Preparando e Treinando Tópicos...
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get 

In [ ]:
# --- CÉLULA FINAL: EMPACOTAMENTO DINÂMICO COMPLETO COM INCLUSÃO DO REPORT K-FOLD ---
import os
import json
import glob
import shutil

%cd /content/ULEEN/software_model

# 1. Captura dinâmica baseada estritamente na sua célula de macros globais
res_bits = BITS_RESOLUTION if 'BITS_RESOLUTION' in globals() else 16
prune_val = PRUNE_THRESHOLD if 'PRUNE_THRESHOLD' in globals() else 0.05
prune_str = str(float(prune_val)).replace('.', '_')

# Captura toda a lista LUT_SIZE e transforma em string (ex: [6, 64, 1] vira "6_64_1")
if 'LUT_SIZE' in globals() and isinstance(LUT_SIZE, (list, tuple)):
    lut_str = "_".join(str(x) for x in LUT_SIZE)
else:
    lut_str = "6_64_1"

# 2. Construção dos nomes dinâmicos baseados nas variáveis completas do teste atual
nome_pasta_experimento = f"experimento_uleen_res{res_bits}_lut{lut_str}_p{prune_str}"
nome_arquivo_zip = f"entregaveis_ton_iot_res{res_bits}_lut{lut_str}_p{prune_str}.zip"

print(f"=== INICIANDO EMPACOTAMENTO DINÂMICO COMPLETO ===")
print(f"[INFO] Configuração detectada: Resolução={res_bits}b | Parâmetros LUT=[{lut_str.replace('_', ', ')}] | Poda={prune_val}")
print(f"[INFO] Gerando arquivo: {nome_arquivo_zip}\n")

pasta_raiz = f"/content/{nome_pasta_experimento}"
pasta_modelos = os.path.join(pasta_raiz, "modelos_podados")
pasta_dados = os.path.join(pasta_raiz, "dados_teste")

if os.path.exists(pasta_raiz):
    shutil.rmtree(pasta_raiz)
os.makedirs(pasta_modelos, exist_ok=True)
os.makedirs(pasta_dados, exist_ok=True)

# 3. Coleta e centralização dos artefatos via Python Puro
modelos_encontrados = glob.glob("**/*pruned*.pt", recursive=True)
for model_path in modelos_encontrados:
    shutil.copy(model_path, pasta_modelos)
    print(f"[COPIADO] Modelo: {os.path.basename(model_path)}")

dados_encontrados = glob.glob("X_test_*.npy") + glob.glob("y_test_*.npy")
for data_path in dados_encontrados:
    shutil.copy(data_path, pasta_dados)
    print(f"[COPIADO] Matriz de Teste: {os.path.basename(data_path)}")

if os.path.exists('/content/relatorio_comparacao_uleen.csv'):
    csv_dinamico = os.path.join(pasta_raiz, f"relatorio_res{res_bits}_lut{lut_str}_p{prune_str}.csv")
    shutil.copy('/content/relatorio_comparacao_uleen.csv', csv_dinamico)
    print(f"[COPIADO] Relatório de Comparação: {os.path.basename(csv_dinamico)}")

# MODIFICAÇÃO INJETADA: Captura automática e renomeação dinâmica do relatório de Validação Cruzada (K-Fold)
kfold_report_local = 'kfold_metrics_report.csv'
if os.path.exists(kfold_report_local):
    csv_kfold_dinamico = os.path.join(pasta_raiz, f"relatorio_kfold_res{res_bits}_lut{lut_str}_p{prune_str}.csv")
    shutil.copy(kfold_report_local, csv_kfold_dinamico)
    print(f"[COPIADO] Relatório Estatístico K-Fold: {os.path.basename(csv_kfold_dinamico)}")

if os.path.exists('/content/auditoria_vies_generic.csv'):
    shutil.copy('/content/auditoria_vies_generic.csv', os.path.join(pasta_raiz, "auditoria_vies_generic.csv"))
    shutil.copy('/content/auditoria_vies_generic.txt', os.path.join(pasta_raiz, "auditoria_vies_generic.txt"))
    print("[COPIADO] Relatórios de auditoria de viés inseridos no pacote!")

# 4. Compactação final do experimento atual
%cd /content
if os.path.exists(nome_arquivo_zip):
    os.remove(nome_arquivo_zip)

os.system(f"zip -q -r {nome_arquivo_zip} {nome_pasta_experimento}")
shutil.rmtree(pasta_raiz)

print(f"\n[SUCESSO INTEGRAL] O arquivo compactado foi gerado com sucesso!")
print(f"-> Caminho final: /content/{nome_arquivo_zip}")
print("Atualize a barra lateral esquerda do seu Colab para baixar o pacote do teste atual.")

/content/ULEEN/software_model
=== INICIANDO EMPACOTAMENTO DINÂMICO COMPLETO ===
[INFO] Configuração detectada: Resolução=16b | Parâmetros LUT=[6, 64, 1] | Poda=0.0005
[INFO] Gerando arquivo: entregaveis_ton_iot_res16_lut6_64_1_p0_0005.zip

[COPIADO] Modelo: model_generic_pruned_0_0005.pt
[COPIADO] Modelo: model_thermostat_pruned_0_0005.pt
[COPIADO] Modelo: model_garage_door_pruned_0_0005.pt
[COPIADO] Modelo: model_gps_tracker_pruned_0_0005.pt
[COPIADO] Modelo: model_modbus_pruned_0_0005.pt
[COPIADO] Modelo: model_weather_pruned_0_0005.pt
[COPIADO] Modelo: model_fridge_pruned_0_0005.pt
[COPIADO] Modelo: model_motion_light_pruned_0_0005.pt
[COPIADO] Matriz de Teste: X_test_modbus.npy
[COPIADO] Matriz de Teste: X_test_thermostat.npy
[COPIADO] Matriz de Teste: X_test_fridge.npy
[COPIADO] Matriz de Teste: X_test_gps_tracker.npy
[COPIADO] Matriz de Teste: X_test_weather.npy
[COPIADO] Matriz de Teste: X_test_motion_light.npy
[COPIADO] Matriz de Teste: X_test_generic.npy
[COPIADO] Matriz de Te